## Mini tutorial: Deploying code and packages to PySpark executors

Prerequisites
- A local or remote Spark cluster with PySpark available.
- Basic familiarity with the Spark driver/executor model.
- For production clusters, permission to configure executor Python environments.

This notebook is the PySpark counterpart to the Dask `code_on_workers.ipynb` example. In Spark, code starts on the driver and runs on executors through jobs. That means environment checks, import checks, and one-off setup are usually expressed as small tasks that run on each executor partition.


In [ ]:
from pathlib import Path
from pprint import pprint

# custom setup modules
from spark_setup import SparkSetup
from setup_data_gaia_dr3 import SetupDataGaiaDR3

### Attach to Spark

For a minimal runnable example, start a local Spark session. On a managed cluster, replace the builder configuration with the cluster defaults or use the Spark session provided by your notebook environment.


In [ ]:
spark_setup = SparkSetup(pods=10, cpu=5, mem=10, data_setup=SetupDataGaiaDR3)
spark = spark_setup.get_spark_session()

sc = spark.sparkContext
print("Spark version:", spark.version)
print("Spark master:", sc.master)
print("Default parallelism:", sc.defaultParallelism)

### Check environment consistency across executors

Goal: collect Python version, executable, hostname, process ID, and platform information from the Python processes that execute Spark tasks. In local mode these may all come from the same machine; on a real cluster this helps identify environment drift across executors.


In [ ]:
def executor_env_info(_):
    import os
    import platform
    import socket
    import sys

    from pyspark import TaskContext

    context = TaskContext.get()
    yield {
        "partition_id": context.partitionId() if context else None,
        "attempt_number": context.attemptNumber() if context else None,
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
        "py_version": sys.version,
        "executable": sys.executable,
        "platform": platform.platform(),
        "pythonpath": os.environ.get("PYTHONPATH", ""),
        "pyspark_python": os.environ.get("PYSPARK_PYTHON", ""),
    }


num_tasks = max(sc.defaultParallelism, 2)
envs = sc.parallelize(range(num_tasks), num_tasks).mapPartitions(executor_env_info).collect()
pprint(envs)


### Verify package imports on executors

Spark does not have a direct equivalent to Dask's `client.run`. To check whether a Python package is available on executors, run a tiny Spark job that imports the package inside each partition.

For reproducible clusters, prefer building the executor environment ahead of time with conda, virtualenv, Docker, cluster images, or Spark archive support. Installing packages from inside tasks is possible in some environments, but it is slower, less reproducible, and often blocked by cluster policy.


In [ ]:
def check_import_on_executor(package_name):
    def _check(_):
        import importlib
        import os
        import socket

        from pyspark import TaskContext

        context = TaskContext.get()
        try:
            module = importlib.import_module(package_name)
            yield {
                "partition_id": context.partitionId() if context else None,
                "hostname": socket.gethostname(),
                "pid": os.getpid(),
                "ok": True,
                "module_file": getattr(module, "__file__", "built-in"),
                "error": "",
            }
        except Exception as exc:
            yield {
                "partition_id": context.partitionId() if context else None,
                "hostname": socket.gethostname(),
                "pid": os.getpid(),
                "ok": False,
                "module_file": "",
                "error": repr(exc),
            }

    return sc.parallelize(range(num_tasks), num_tasks).mapPartitions(_check).collect()


# Replace this with the import name you need to verify, for example "zero_point".
package_name = "json"
import_results = check_import_on_executor(package_name)
pprint(import_results)


### Optional: install a package on executors during a task

Use this only for short-lived experiments where your cluster policy allows network access from executors. For shared or production workflows, bake dependencies into the executor environment instead.

The cell below is intentionally not executed by default. Uncomment it only when you understand the cost and reproducibility tradeoffs.


In [ ]:
# def install_package_on_executor(package_spec):
#     def _install(_):
#         import os
#         import socket
#         import subprocess
#         import sys
#
#         from pyspark import TaskContext
#
#         context = TaskContext.get()
#         subprocess.check_call([sys.executable, "-m", "pip", "install", package_spec])
#         yield {
#             "partition_id": context.partitionId() if context else None,
#             "hostname": socket.gethostname(),
#             "pid": os.getpid(),
#             "installed": package_spec,
#         }
#
#     return sc.parallelize(range(num_tasks), num_tasks).mapPartitions(_install).collect()
#
# install_results = install_package_on_executor("gaiadr3-zeropoint")
# pprint(install_results)


### Upload your own code to executors

Use `SparkContext.addPyFile` to distribute a local `.py`, `.zip`, or `.egg` file to executors. The file is added to the Python path for tasks, so executor code can import it by module name. For production jobs, the same idea is usually expressed with `spark-submit --py-files` or the `spark.submit.pyFiles` configuration.


In [ ]:
helper_path = Path("astroflow_worker_helper.py")
helper_path.write_text(
    "def describe_source(source_id):\n"
    "    return f'processed-source-{source_id}'\n",
    encoding="utf-8",
)

sc.addPyFile(str(helper_path.resolve()))
print("Uploaded helper module:", helper_path.resolve())


### Verify the uploaded helper on executors

This small job imports the helper module inside executor tasks and applies it to synthetic source IDs. If the import succeeds, the uploaded file is visible to executor-side Python code.


In [ ]:
def use_uploaded_helper(source_id):
    from astroflow_worker_helper import describe_source

    return describe_source(source_id)


source_ids = list(range(8))
helper_results = sc.parallelize(source_ids, 4).map(use_uploaded_helper).collect()
pprint(helper_results)


### Cleanup

Stop the local Spark session when the notebook is complete. If you are using a managed notebook service that owns the Spark session, follow that environment's guidance instead.


In [ ]:
spark.stop()


Tips and best practices
- Prefer prebuilt executor environments over installing packages from inside Spark tasks.
- Pin versions for Python packages and Spark dependencies so driver and executor environments do not drift.
- Use `SparkContext.addPyFile`, `spark-submit --py-files`, or `spark.submit.pyFiles` for small Python modules that should travel with the job.
- Use `--packages` or `spark.jars.packages` for JVM-side Spark dependencies, not Python packages.
- For larger Python environments, use cluster-supported mechanisms such as conda-pack, virtualenv archives, Docker images, or managed runtime images.
- Keep executor-side setup idempotent. Spark may retry failed tasks, so setup code can run more than once.
- Be careful with security: executor-side code runs across your cluster. Only distribute code and dependencies from trusted sources.
- Use the Spark UI to confirm task placement, failures, retries, and executor behavior when debugging environment issues.
